In [0]:
customers_df = spark.read.table("global_wholesale_distributor.staging_wholesale_distributor.sl_customers")
exchange_rates_df = spark.read.table("global_wholesale_distributor.staging_wholesale_distributor.sl_exchange_rates")
invoice_line_items_df = spark.read.table("global_wholesale_distributor.staging_wholesale_distributor.sl_invoice_line_items")
invoices_df = spark.read.table("global_wholesale_distributor.staging_wholesale_distributor.sl_invoices")
payments_df = spark.read.table("global_wholesale_distributor.staging_wholesale_distributor.sl_payments")
products_df = spark.read.table("global_wholesale_distributor.staging_wholesale_distributor.sl_products")
regions_df = spark.read.table("global_wholesale_distributor.staging_wholesale_distributor.sl_regions")

In [0]:
from pyspark.sql.functions import col, round

payments_with_currency = payments_df.drop("original_currency").join(
    invoices_df.select("invoice_id", "currency"), 
    "invoice_id", 
    "inner"
)

payments_with_currency = payments_with_currency.withColumnRenamed("currency", "original_currency")

payments_usd = payments_with_currency.join(
    exchange_rates_df,
    (col("original_currency") == exchange_rates_df.currency) & 
    (col("payment_date") == exchange_rates_df.date),
    "left"
)

payments_usd = payments_usd.withColumn(
    "payment_amount_usd",
    round(col("payment_amount") * col("rate_to_usd"), 2)
)

payments_usd = payments_usd.select(
    "payment_id",
    "invoice_id",
    "payment_date",
    "payment_amount",
    "original_currency", 
    "payment_amount_usd",
    "payment_method"
)

display(payments_usd)

In [0]:
from pyspark.sql.functions import col, round, to_date

products_with_invoice = products_df.drop("original_currency").join(
    invoice_line_items_df.select("product_id", "invoice_id"),
    "product_id",
    "inner"
).join(
    invoices_df.select("invoice_id", "currency", "invoice_date"),
    "invoice_id",
    "inner"
)

products_with_invoice = products_with_invoice.withColumnRenamed("currency", "original_currency")

products_with_currency = products_with_invoice.join(
    exchange_rates_df,
    (col("original_currency") == exchange_rates_df.currency) & 
    (col("invoice_date") == to_date(exchange_rates_df.date, "yyyy/M/d")),
    "left"
)

products_usd = products_with_currency.withColumn(
    "cost_price_usd",
    round(col("cost_price") * col("rate_to_usd"), 2)
)

products_usd = products_usd.select(
    "product_id",
    "product_name",
    "category",
    "cost_price",
    "original_currency",
    "cost_price_usd",
)

display(products_usd)

In [0]:
from pyspark.sql.functions import col, round

invoice_line_items_with_currency = invoice_line_items_df.drop("original_currency").join(
    invoices_df.select("invoice_id", "currency", "invoice_date"),
    "invoice_id",
    "inner"
).withColumnRenamed("currency", "original_currency")

invoice_line_items_with_currency = invoice_line_items_with_currency.join(
    exchange_rates_df,
    (col("original_currency") == exchange_rates_df.currency) & 
    (col("invoice_date") == exchange_rates_df.date),
    "left"
)

invoice_line_items_usd = invoice_line_items_with_currency.withColumn(
    "unit_price_usd", 
    round(col("unit_price") * col("rate_to_usd"), 2)
)

invoice_line_items_usd = invoice_line_items_usd.select(
    "invoice_id",
    "invoice_line_id",
    "product_id",
    "quantity",
    "unit_price",
    "discount",
    "original_currency",
    "unit_price_usd"
)
display(invoice_line_items_usd)

In [0]:
catalog_name = "global_wholesale_distributor"
schema_name = "staging_wholesale_distributor"
table_name = "sl_payments"
table_path = f"{catalog_name}.{schema_name}.{table_name}"

payments_usd.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(table_path)

In [0]:
catalog_name = "global_wholesale_distributor"
schema_name = "staging_wholesale_distributor"
table_name = "sl_invoice_line_items"
table_path = f"{catalog_name}.{schema_name}.{table_name}"

invoice_line_items_usd.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(table_path)

In [0]:
catalog_name = "global_wholesale_distributor"
schema_name = "staging_wholesale_distributor"
table_name = "sl_products"
table_path = f"{catalog_name}.{schema_name}.{table_name}"

products_usd.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(table_path)